In [1]:
from obtain_allocation import *
from study_results_per_line import *
from generate_gaussian import *
import os
import pandas as pd
import numpy as np
import matplotlib as plt

# ANALISIS toA

In [2]:
def generate_gaussian_per_slot_all_days(dataframeinput: pd.DataFrame, param_to_gauss: str):
    """
    Analyzes 'ALL' workdays. If initial normality fails, it filters outliers
    and replaces the results with the cleaned dataset statistics.
    """

    dataframe = dataframeinput[(dataframeinput["Occupation"] <= 80) & (dataframeinput["Occupation"] >= 0)]
    dataframe = dataframe[(dataframe["DayType"] == "LA") & (dataframe["FreeDay"] == 0)]

    if dataframe.empty:
        return pd.DataFrame()

    stopnum = dataframe["Stop"].iloc[0]
    timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]

    columns = ["day", "Stop"]
    for time in timeslot:
        columns.extend([
            f"Mean_{time}",
            f"std_{time}",
            f"IsNormal_{time}",
            f"outliers_{time}",
            f"instances_{time}"
        ])

    row_data = ["ALL", stopnum]

    for time in timeslot:
        data_per_timeslot = dataframe[dataframe["TimeSlot"] == time].copy()

        gauss_hypothesis, instances_involved = is_normal_modified(
            data_per_timeslot, "ALL", time, param_to_gauss, False
        )

        outliers_found = []

        if not isinstance(instances_involved, list) and not instances_involved.empty:
            current_values = instances_involved[param_to_gauss].tolist()
            n = int(0.01*len(instances_involved))
            outliers_found = estimate_outliers(current_values, n, False)

            if not gauss_hypothesis and len(outliers_found) > 0:
                # Filter out the outliers
                indices_to_drop = []
                temp_values = data_per_timeslot[param_to_gauss].tolist()
                for outlier in outliers_found:
                    if outlier in temp_values:
                        idx = data_per_timeslot.index[temp_values.index(outlier)]
                        indices_to_drop.append(idx)
                        temp_values[temp_values.index(outlier)] = None  # mark as consumed

                data_cleaned = data_per_timeslot.drop(index=indices_to_drop)

                gauss_hypothesis, instances_involved = is_normal_modified(
            data_cleaned, "ALL", time, param_to_gauss, False
            )

        # --- Final Stats Calculation ---
        if not isinstance(instances_involved, list) and not instances_involved.empty:
            final_values = instances_involved[param_to_gauss].tolist()

            mean_val = np.mean(final_values)
            std_val = sqrt(central_moment(final_values, 2))
            instances_list = instances_involved["Occupation"].tolist()

            row_data.extend([mean_val, std_val, gauss_hypothesis, outliers_found, instances_list])
        else:
            row_data.extend([0, 0, "NO DATA", 0, 0])

    new_dataframe = pd.DataFrame([row_data], columns=columns)
    return new_dataframe


def generate_gaussian_per_slot_days(dataframeinput: pd.DataFrame, param_to_gauss: str):
    """
    Analyzes given workdays. If initial normality fails, it filters outliers
    and replaces the results with the cleaned dataset statistics.
    """

    dataframe = dataframeinput[(dataframeinput["Occupation"] <= 80) & (dataframeinput["Occupation"] >= 0)]
    dataframe = dataframe[(dataframe["DayType"] == "LA") & (dataframe["FreeDay"] == 0)]

    if dataframe.empty:
        return pd.DataFrame()

    stopnum = dataframe["Stop"].iloc[0]
    timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]

    columns = ["day", "Stop"]
    for time in timeslot:
        columns.extend([
            f"Mean_{time}",
            f"std_{time}",
            f"IsNormal_{time}",
            f"outliers_{time}",
            f"instances_{time}"
        ])

    rows = []
    for day in ["L", "M", "X", "J", "V"]:
        row_data = [day, stopnum]

        for time in timeslot:
            data_per_timeslot = dataframe[dataframe["TimeSlot"] == time].copy()

            gauss_hypothesis, instances_involved = is_normal_modified(
                data_per_timeslot, day, time, param_to_gauss, False
            )

            outliers_found = []

            if not isinstance(instances_involved, list) and not instances_involved.empty:
                current_values = instances_involved[param_to_gauss].tolist()
                n = int(0.01*len(instances_involved))
                outliers_found = estimate_outliers(current_values, n, False)

                if not gauss_hypothesis and len(outliers_found) > 0:
                    # Filter out the outliers
                    indices_to_drop = []
                    temp_values = data_per_timeslot[param_to_gauss].tolist()
                    for outlier in outliers_found:
                        if outlier in temp_values:
                            idx = data_per_timeslot.index[temp_values.index(outlier)]
                            indices_to_drop.append(idx)
                            temp_values[temp_values.index(outlier)] = None  # mark as consumed

                    data_cleaned = data_per_timeslot.drop(index=indices_to_drop)

                    gauss_hypothesis, instances_involved = is_normal_modified(
                data_cleaned, day, time, param_to_gauss, False
                )

            # --- Final Stats Calculation ---
            if not isinstance(instances_involved, list) and not instances_involved.empty:
                final_values = instances_involved[param_to_gauss].tolist()

                mean_val = np.mean(final_values)
                std_val = sqrt(central_moment(final_values, 2))
                instances_list = instances_involved["Occupation"].tolist()

                row_data.extend([mean_val, std_val, gauss_hypothesis, outliers_found, instances_list])
            else:
                row_data.extend([0, 0, "NO DATA", 0, 0])
        rows.append(row_data) 

    new_dataframe = pd.DataFrame(rows, columns=columns)
    return new_dataframe

In [3]:
files_stops = files = os.listdir("../data/E01_toA/NEW")


#Creamos plantilla para el dataframe FINAL

daytype = ["L", "M", "X", "J", "V"]
timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


#Inittialy column day to specify L, M, X, J, V
columns = ["day"]
columns.append("Stop")

#For each timeslot t add columns Mean_t, std_t and IsNormal_t
for time in timeslot:
    strvar = f"Mean_{time}"
    columns.append(strvar)
    strvar = f"std_{time}"
    columns.append(strvar)
    strvar = f"IsNormal_{time}"
    columns.append(strvar)
    strvar = f"outliers_{time}"
    columns.append(strvar)
    strvar = f"instances_{time}"
    columns.append(strvar)

#generate df
final_dataframe = pd.DataFrame(columns=columns)
final_dataframe_daily = pd.DataFrame(columns=columns)


# Loop over files
for file in files_stops:
    if file != "RESULTS" and file != "E01_toA.zip":
        dataframe = pd.read_csv("../data/E01_toA/NEW/" + file, encoding="utf-8", header=0)
        dataframe = clean_dataframe(dataframe, 2)
        new_dataframe = generate_gaussian_per_slot_all_days(dataframe, "Occupation")
        
        new_dataframe_daily = generate_gaussian_per_slot_days(dataframe, "Occupation")
        if not new_dataframe_daily.empty:
            final_dataframe_daily = pd.concat([final_dataframe_daily, new_dataframe_daily], ignore_index=True)
        # Using concat as append is deprecated in newer pandas versions
        if not new_dataframe.empty:
            final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)
        
        

# ---------------------------------------------------------
# NEW: CALCULATE AND PRINT THE PERCENTAGE OF TRUE NORMALITY
# ---------------------------------------------------------
is_normal_cols = [col for col in final_dataframe.columns if "IsNormal_" in col]

# Flatten all the IsNormal columns into a single 1D array to count them easily
normality_values = final_dataframe[is_normal_cols].values.flatten()


total_true, total_tested = 0, 0

# Count how many are strictly True
total_true = (normality_values == True).sum()

total_nd = (normality_values == "NO DATA").sum()
# Count how many actually had data to test (excluding "NO DATA")
total_tested = (normality_values != "NO DATA").sum()

if total_tested > 0:
    percentage_true = (total_true / total_tested) * 100
    print(f"--- NORMALITY RESULTS ---")
    print(f"Total combinations tested: {total_tested}")
    print(f"Normal distributions (TRUE): {total_true}")
    print(f"Normal distributions (NO DATA): {total_nd}")
    print(f"Percentage of TRUE normality: {percentage_true:.2f}%")
else:
    print("No valid combinations were tested for normality.")
# ---------------------------------------------------------



is_normal_cols = [col for col in final_dataframe_daily.columns if "IsNormal_" in col]

# Flatten all the IsNormal columns into a single 1D array to count them easily
normality_values = final_dataframe_daily[is_normal_cols].values.flatten()


total_true, total_tested = 0, 0

# Count how many are strictly True
total_true = (normality_values == True).sum()

total_nd = (normality_values == "NO DATA").sum()
# Count how many actually had data to test (excluding "NO DATA")
total_tested = (normality_values != "NO DATA").sum()

if total_tested > 0:
    percentage_true = (total_true / total_tested) * 100
    print(f"--- NORMALITY RESULTS ---")
    print(f"Total combinations tested: {total_tested}")
    print(f"Normal distributions (TRUE): {total_true}")
    print(f"Normal distributions (NO DATA): {total_nd}")
    print(f"Percentage of TRUE normality: {percentage_true:.2f}%")
else:
    print("No valid combinations were tested for normality.")


with open(f"../data/E01_toA/NEW/RESULTS/FINAL_results_NORMAL.csv", "w", encoding="utf-8") as f:
        final_dataframe.to_csv(f, index=False)

with open(f"../data/E01_toA/NEW/RESULTS/FINAL_results_with_daily.csv", "w", encoding="utf-8") as f:
        final_dataframe_daily.to_csv(f, index=False)

final_final_dataframe = pd.concat([final_dataframe, final_dataframe_daily], ignore_index=True)
print(analyze_normality_results_v2(final_final_dataframe,'toA',"../data/E01_toA/NEW/RESULTS/PLOTS"))


/tmp/ipykernel_14730/1498732781.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe_daily = pd.concat([final_dataframe_daily, new_dataframe_daily], ignore_index=True)
/tmp/ipykernel_14730/1498732781.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)
/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/generate_gaussian.py:260: RuntimeWarning: invalid value encountered in scalar divide
  retur

--- NORMALITY RESULTS ---
Total combinations tested: 285
Normal distributions (TRUE): 269
Normal distributions (NO DATA): 0
Percentage of TRUE normality: 94.39%
--- NORMALITY RESULTS ---
Total combinations tested: 1423
Normal distributions (TRUE): 1060
Normal distributions (NO DATA): 2
Percentage of TRUE normality: 74.49%


/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/study_results_per_line.py:192: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{t}:00" for t in dataframe_kpi['Timeslot']], rotation=45)
/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/study_results_per_line.py:192: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{t}:00" for t in dataframe_kpi['Timeslot']], rotation=45)


{'norm_timeslot_all':     Timeslot    True_pct  False_pct  Total
0          5   93.333333   6.666667     15
1          6   93.333333   6.666667     15
2          7   93.333333   6.666667     15
3          8   86.666667  13.333333     15
4          9   93.333333   6.666667     15
5         10   93.333333   6.666667     15
6         11  100.000000   0.000000     15
7         12  100.000000   0.000000     15
8         13   93.333333   6.666667     15
9         14   93.333333   6.666667     15
10        15  100.000000   0.000000     15
11        16  100.000000   0.000000     15
12        17  100.000000   0.000000     15
13        18  100.000000   0.000000     15
14        19  100.000000   0.000000     15
15        20  100.000000   0.000000     15
16        21  100.000000   0.000000     15
17        22  100.000000   0.000000     15
18        23   53.333333  46.666667     15, 'outliers_timeslot_all':     Timeslot  Avg_outliers  Std_outliers  Total_outliers
0          5      0.533333      0.9

# toB

In [4]:
files_stops = files = os.listdir("../data/E01_toB")


#Creamos plantilla para el dataframe FINAL

daytype = ["L", "M", "X", "J", "V"]
timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


#Inittialy column day to specify L, M, X, J, V
columns = ["day"]
columns.append("Stop")

#For each timeslot t add columns Mean_t, std_t and IsNormal_t
for time in timeslot:
    strvar = f"Mean_{time}"
    columns.append(strvar)
    strvar = f"std_{time}"
    columns.append(strvar)
    strvar = f"IsNormal_{time}"
    columns.append(strvar)
    strvar = f"outliers_{time}"
    columns.append(strvar)
    strvar = f"instances_{time}"
    columns.append(strvar)

#generate df
final_dataframe = pd.DataFrame(columns=columns)
final_dataframe_daily = pd.DataFrame(columns=columns)

# Loop over files
for file in files_stops:
    if file != "RESULTS" and file != "E01_toB.zip":
        dataframe = pd.read_csv("../data/E01_toB/" + file, encoding="utf-8", header=0)
        dataframe = clean_dataframe(dataframe, 1)
        new_dataframe = generate_gaussian_per_slot_all_days(dataframe, "Occupation")
        new_dataframe_daily = generate_gaussian_per_slot_days(dataframe, "Occupation")
        if not new_dataframe_daily.empty:
            final_dataframe_daily = pd.concat([final_dataframe_daily, new_dataframe_daily], ignore_index=True)
        # Using concat as append is deprecated in newer pandas versions
        if not new_dataframe.empty:
            final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)
# ---------------------------------------------------------
# CALCULATE AND PRINT THE PERCENTAGE OF TRUE NORMALITY
# ---------------------------------------------------------
is_normal_cols = [col for col in final_dataframe.columns if "IsNormal_" in col]

# Flatten all the IsNormal columns into a single 1D array to count them easily
normality_values = final_dataframe[is_normal_cols].values.flatten()

total_true, total_tested = 0, 0

# Count how many are strictly True
total_true = (normality_values == True).sum()

total_nd = (normality_values == "NO DATA").sum()
# Count how many actually had data to test (excluding "NO DATA")
total_tested = (normality_values != "NO DATA").sum()

if total_tested > 0:
    percentage_true = (total_true / total_tested) * 100
    print(f"--- NORMALITY RESULTS ---")
    print(f"Total combinations tested: {total_tested}")
    print(f"Normal distributions (TRUE): {total_true}")
    print(f"Normal distributions (NO DATA): {total_nd}")
    print(f"Percentage of TRUE normality: {percentage_true:.2f}%")
else:
    print("No valid combinations were tested for normality.")
# ---------------------------------------------------------
# ---------------------------------------------------------



is_normal_cols = [col for col in final_dataframe_daily.columns if "IsNormal_" in col]

# Flatten all the IsNormal columns into a single 1D array to count them easily
normality_values = final_dataframe_daily[is_normal_cols].values.flatten()


total_true, total_tested = 0, 0

# Count how many are strictly True
total_true = (normality_values == True).sum()

total_nd = (normality_values == "NO DATA").sum()
# Count how many actually had data to test (excluding "NO DATA")
total_tested = (normality_values != "NO DATA").sum()

if total_tested > 0:
    percentage_true = (total_true / total_tested) * 100
    print(f"--- NORMALITY RESULTS ---")
    print(f"Total combinations tested: {total_tested}")
    print(f"Normal distributions (TRUE): {total_true}")
    print(f"Normal distributions (NO DATA): {total_nd}")
    print(f"Percentage of TRUE normality: {percentage_true:.2f}%")
else:
    print("No valid combinations were tested for normality.")

with open(f"../data/E01_toB/RESULTS/FINAL_results_NORMAL.csv", "w", encoding="utf-8") as f:
        final_dataframe.to_csv(f, index=False)

with open(f"../data/E01_toB/RESULTS/FINAL_results_with_daily.csv", "w", encoding="utf-8") as f:
        final_dataframe_daily.to_csv(f, index=False)

final_final_dataframe = pd.concat([final_dataframe, final_dataframe_daily], ignore_index=True)
print(analyze_normality_results_v2(final_final_dataframe, 'toB',"../data/E01_toB/RESULTS/PLOTS"))

/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/generate_gaussian.py:260: RuntimeWarning: invalid value encountered in scalar divide
  return central_moment(x, 3) / central_moment(x, 2)**(3/2)
/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/generate_gaussian.py:257: RuntimeWarning: invalid value encountered in scalar divide
  return (central_moment(x, 4) / central_moment(x, 2) ** 2) -3
/tmp/ipykernel_14730/1761446913.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe_daily = pd.concat([final_dataframe_daily, new_dataframe_daily], ignore_index=True)
/tmp/ipykernel_14730/1761446913.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version

--- NORMALITY RESULTS ---
Total combinations tested: 228
Normal distributions (TRUE): 202
Normal distributions (NO DATA): 0
Percentage of TRUE normality: 88.60%
--- NORMALITY RESULTS ---
Total combinations tested: 1137
Normal distributions (TRUE): 772
Normal distributions (NO DATA): 3
Percentage of TRUE normality: 67.90%


/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/study_results_per_line.py:192: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{t}:00" for t in dataframe_kpi['Timeslot']], rotation=45)
/home/hectoralv22/Desktop/CATEDRA/REPO/GIAA_EMT/DATOS/src/study_results_per_line.py:192: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{t}:00" for t in dataframe_kpi['Timeslot']], rotation=45)


{'norm_timeslot_all':     Timeslot    True_pct  False_pct  Total
0          5   75.000000  25.000000     12
1          6   91.666667   8.333333     12
2          7  100.000000   0.000000     12
3          8   91.666667   8.333333     12
4          9   91.666667   8.333333     12
5         10   91.666667   8.333333     12
6         11   91.666667   8.333333     12
7         12   91.666667   8.333333     12
8         13  100.000000   0.000000     12
9         14  100.000000   0.000000     12
10        15   91.666667   8.333333     12
11        16   91.666667   8.333333     12
12        17  100.000000   0.000000     12
13        18   91.666667   8.333333     12
14        19   91.666667   8.333333     12
15        20   91.666667   8.333333     12
16        21   91.666667   8.333333     12
17        22   91.666667   8.333333     12
18        23   16.666667  83.333333     12, 'outliers_timeslot_all':     Timeslot  Avg_outliers  Std_outliers  Total_outliers
0          5      0.666667      0.4

As the results are more than accurate for assumming the normality of the occupation value. We store, for every stop in every timeslot in both Directions, we add just the mean and std.

In [5]:
dataframe_toA = pd.read_csv("../data/E01_toA/NEW/RESULTS/FINAL_results_NORMAL.csv")

columns =["Stop"]

for time in timeslot:
    columns.append(f"Mean_{time}")
    columns.append(f"std_{time}")



final_export_dataframe_toA = dataframe_toA[columns].copy()


dataframe_toB = pd.read_csv("../data/E01_toB/RESULTS/FINAL_results_NORMAL.csv")


columns =["Stop"]

for time in timeslot:
    columns.append(f"Mean_{time}")
    columns.append(f"std_{time}")



final_export_dataframe_toB = dataframe_toB[columns].copy()


with open(f"../../WEB_DEVEL/data/known_data/NORMAL_OCCUPATION_toA.csv", "w") as f:
    final_export_dataframe_toA.to_csv(f, index=False)

with open(f"../../WEB_DEVEL/data/known_data/NORMAL_OCCUPATION_toB.csv", "w") as f:
    final_export_dataframe_toB.to_csv(f, index=False)